# Rainfall Analysis and Planting Advisory

This notebook cleans the Machakos station readings, analyses rainfall across five Kenyan counties with NumPy, and exports a county-level summary for the NGO. Rainfall values are in millimetres.

## Clean the Machakos readings

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

def parse_rainfall(value):
    try:
        return int(value.strip())
    except ValueError:
        return None

parsed_machakos = [parse_rainfall(value) for value in machakos_raw]
valid_values = [value for value in parsed_machakos if value is not None]
machakos_imputation = round(sum(valid_values) / len(valid_values))
imputed_months = [month for month, value in zip(months, parsed_machakos) if value is None]
machakos = [machakos_imputation if value is None else value for value in parsed_machakos]

print("Machakos valid-value average (rounded):", machakos_imputation, "mm")
print("Imputed months:", ", ".join(imputed_months))
print("Cleaned Machakos row:", machakos)

Machakos valid-value average (rounded): 68 mm
Imputed months: April, September
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Combine counties and inspect the array

In [2]:
R = np.array(clean_rows + [machakos])
print("Rainfall array:\n", R)
print("Shape:", R.shape)
print("Dtype:", R.dtype)

Rainfall array:
 [[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Dtype: int64


## Annual totals and monthly averages

In [3]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)
print("Annual total rainfall by county (mm):")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total}")
print("\nAverage rainfall by month across all counties (mm):")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f}")

Annual total rainfall by county (mm):
Nairobi: 1045
Nakuru: 916
Kisumu: 1371
Mombasa: 1075
Machakos: 811

Average rainfall by month across all counties (mm):
January: 47.0
February: 47.6
March: 91.0
April: 164.4
May: 171.2
June: 67.6
July: 56.6
August: 58.2
September: 64.2
October: 67.8
November: 127.4
December: 80.6


## Wettest month per county

In [4]:
wettest_indices = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_indices]
for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")

Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


## Dry months below 50 mm

In [5]:
dry_month_counts = (R < 50).sum(axis=1)
for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count}")

Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


## Recommended two-month planting windows

In [6]:
consecutive_totals = R[:, :-1] + R[:, 1:]
window_starts = np.argmax(consecutive_totals, axis=1)
planting_windows = [f"{months[index]}-{months[index + 1]}" for index in window_starts]
window_rainfall = consecutive_totals[np.arange(R.shape[0]), window_starts]
for county, window, total in zip(counties, planting_windows, window_rainfall):
    print(f"{county}: {window} ({total} mm combined)")

Nairobi: April-May (395 mm combined)
Nakuru: April-May (265 mm combined)
Kisumu: April-May (420 mm combined)
Mombasa: April-May (390 mm combined)
Machakos: November-December (278 mm combined)


## Export the summary

In [7]:
summary_rows = zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows)
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    writer.writerows(summary_rows)
print("Wrote rainfall_summary.csv")

Wrote rainfall_summary.csv


## Planting Advisory

Use the county-specific two-month windows above as initial planting guidance, prioritising the consecutive months with the highest combined rainfall in each station record. Nairobi, Nakuru, Kisumu, and Mombasa all have their strongest adjacent rainfall period in April-May, while Machakos peaks in November-December; confirm each recommendation against the displayed results before distributing SMS messages. The driest-month counts also suggest where farmers may need water-conservation measures or access to supplemental irrigation. These windows describe historical monthly totals only and are not a forecast of the coming season. The Machakos April and September readings were missing or invalid and were replaced with the rounded average of its ten valid readings, so those estimates may affect its totals and recommended window. Verify the source records and incorporate local planting calendars and current forecasts before sending advice.

## Data Quality Notes

Machakos contained two unusable entries: "N/A" for April and a blank value for September. Both were imputed as 68 mm, the rounded mean of the ten valid Machakos readings (67.5 mm before rounding). Mean imputation preserves a complete 5 x 12 array but hides the actual rainfall in those months and may distort the county's annual total and planting window. The analysis assumes all station readings use the same units, monthly periods, and measurement standards; station-level validation and checks against nearby observations are recommended.